# LUT-CHINA 2026: Multi-Scale Modeling Interactive Tutorial

Welcome to the official interactive tutorial for the **LUT-CHINA 2026 iGEM Modeling Framework**.
This notebook walks you step-by-step through running, inspecting, and customizing each of the 4 quantitative modeling layers:
1. **Supramolecular Ternary Micelle Thermodynamics** (Holland-Rubingh Theory & Perrin Inversion)
2. **Bi-Directional Hydrogel Mass Transport** (PDE Discretization & Péclet Decoupling Analysis)
3. **Metabolic Flux & Ordered Bi-Bi Enzyme Kinetics** (Coupled ODE Integration)
4. **Diabetic Wound Contraction PK/PD Dynamics** (Dual-Target Synergistic Hill Model)

---

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path so core modules can be imported directly
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from core.micelle_assembly import HollandRubinghModel, PerrinFrictionModel
from core.mass_transport import HydrogelTransportPDE, PecletDecouplingSolver
from core.metabolic_kinetics import BiBiKineticsODE
from core.wound_pkpd import WoundContractionModel

print("LUT-CHINA 2026 Modeling Suite loaded successfully!")

## 1. Microscopic Scale: Ternary Micelle Assembly Thermodynamics
Let's calculate the mixed Critical Micelle Concentration (CMC) and excess Gibbs free energy $\Delta G^{ex}$.

In [ ]:
with open('../configs/thermodynamics.json', 'r') as f:
    cfg_thermo = json.load(f)

cmc_dict = cfg_thermo['single_cmc_mM']
thermo = HollandRubinghModel(single_cmc=cmc_dict, temperature_k=cfg_thermo['temperature_K'])
chi = [
    cfg_thermo['fitted_interaction_parameters']['chi_AB'],
    cfg_thermo['fitted_interaction_parameters']['chi_AC'],
    cfg_thermo['fitted_interaction_parameters']['chi_BC']
]

alpha = np.array(cfg_thermo['feed_mole_fraction'])
x_micelle = np.array([0.42, 0.38, 0.20])

cmc_mix = thermo.predict_mixed_cmc(alpha, x_micelle, chi)
gibbs = thermo.calculate_gibbs_free_energy(cmc_mix * 1e-3, x_micelle, chi)

print(f"Calculated Mixed CMC: {cmc_mix:.4f} mM")
print(f"Excess Gibbs Free Energy (Delta G_ex): {gibbs['delta_G_excess']:.3f} kJ/mol (< 0 indicates strong synergy)")

# Perrin cylinder length inversion
perrin = PerrinFrictionModel(temperature_K=cfg_thermo['temperature_K'])
L_cyl = perrin.invert_cylinder_length(target_Rh_nm=293.0, diameter_d_nm=3.2)
print(f"Inverted rigid cylinder length L: {L_cyl:.1f} nm (shrunk by 74.5% compared to initial 1150 nm loose aggregates)")

## 2. Mesoscopic Scale: Bi-Directional Hydrogel Mass Transport
Let's solve the concentration gradient across the 3.5 mm dressing and evaluate the effective Péclet number under exudate convective velocity.

In [ ]:
with open('../configs/hydrogel_transport.json', 'r') as f:
    cfg_hydro = json.load(f)

L_mm = cfg_hydro['dressing_thickness_mm']
pde = HydrogelTransportPDE(
    L_mm=L_mm,
    D_top=cfg_hydro['diffusivity_top_m2_s'],
    D_bottom=cfg_hydro['diffusivity_bottom_m2_s'],
    exponent_n=cfg_hydro['exponent_n']
)

z_mm, C_static = pde.solve_steady_state(u_convective=0.0)
_, C_dynamic = pde.solve_steady_state(u_convective=cfg_hydro['upward_fluid_velocity_m_s'])

peclet = PecletDecouplingSolver.calculate_peclet(
    u_m_s=cfg_hydro['upward_fluid_velocity_m_s'],
    L_m=L_mm * 1e-3,
    D_eff_m2_s=cfg_hydro['diffusivity_bottom_m2_s'],
    K_partition=cfg_hydro['partition_coefficient_K'],
    porosity_phi=cfg_hydro['hydrogel_porosity_phi']
)

print(f"Retardation factor R: {peclet['retardation_factor_R']:.1f}")
print(f"Effective Peclet number Pe(eff): {peclet['Pe_effective']:.4f}")
print(f"Is diffusion dominated? {peclet['is_diffusion_dominated_decoupled']} (Pe(eff) << 0.1)")

plt.figure(figsize=(6, 4))
plt.plot(z_mm * 1000, C_static, 'b-', label='Static (u=0)')
plt.plot(z_mm * 1000, C_dynamic, 'r--', label='With Exudate Convection')
plt.xlabel('Dressing Depth z (mm)')
plt.ylabel('Concentration C / C0')
plt.title('Hydrogel Steady-State Gradient Profile')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Cellular Scale: Enzymatic Glycosylation Flux
Let's integrate the Ordered Bi-Bi dynamic equations over 24 hours of induction.

In [ ]:
with open('../configs/metabolic_flux.json', 'r') as f:
    cfg_flux = json.load(f)

bibi = BiBiKineticsODE(
    Vmax=cfg_flux['k_cat_s_inv'] * 3600 * 1e-3 * 0.5,
    Km_UDP=cfg_flux['Km_UDP_Glc_uM'] / 1000.0,
    Km_Aglycone=cfg_flux['Km_calycosin_uM'] / 1000.0
)

t_span = np.linspace(0, 24, 200)
t_out, y_sol = bibi.simulate(t_span, y0=[2.5, 5.0, 0.0])

plt.figure(figsize=(6, 4))
plt.plot(t_out, y_sol[:, 0], label='[UDP-Glc] precursor (mM)', color='#ff7f0e', linestyle='-.')
plt.plot(t_out, y_sol[:, 1], label='[Aglycone] substrate (mM)', color='gray', linestyle='--')
plt.plot(t_out, y_sol[:, 2], label='[Glycoside] engineered product (mM)', color='#1f77b4', lw=2.2)
plt.xlabel('Time (h)')
plt.ylabel('Concentration (mM)')
plt.title('Enzymatic Glycosylation Time-Course')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f"Final Glycoside Product Yield: {y_sol[-1, 2]:.2f} mM")

## 4. Tissue & Clinical Scale: Diabetic Wound Contraction PK/PD
Compare wound closure rates across untreated control, standard commercial dressing, and our dual-gradient nano-hydrogel.

In [ ]:
with open('../configs/wound_healing.json', 'r') as f:
    cfg_pkpd = json.load(f)

wound = WoundContractionModel(
    k_close=cfg_pkpd['baseline_closure_rate_day_inv'],
    alpha=cfg_pkpd['synergy_weight_alpha'],
    beta=cfg_pkpd['synergy_weight_beta'],
    EC50_1=cfg_pkpd['calycosin_angiogenesis_EC50_uM'],
    EC50_2=cfg_pkpd['lobetyolin_anti_inflammatory_EC50_uM']
)

t_days, w_ctrl = wound.simulate_closure(days=21, treatment_type="untreated")
_, w_comm = wound.simulate_closure(days=21, treatment_type="commercial")
_, w_grad = wound.simulate_closure(days=21, treatment_type="dual_gradient")

plt.figure(figsize=(7, 4.5))
plt.plot(t_days, w_ctrl, 'k:', label='Untreated Control')
plt.plot(t_days, w_comm, 'b--', label='Commercial Standard')
plt.plot(t_days, w_grad, color='#d62728', lw=2.5, label='LUT-CHINA Dual-Gradient Hydrogel')
plt.axhline(50, color='gray', linestyle='--', alpha=0.5, label='50% Closure Landmark')
plt.xlabel('Healing Days')
plt.ylabel('Remaining Wound Area (%)')
plt.title('Diabetic Wound Area Closure Dynamics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()